In [1]:
import qiskit
print(qiskit.__version__)

2.4.1


In [2]:
#Pour plotter avec Plotly to png
!pip install kaleido


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# My initial version : left apart to match general grover circuit structure
# #This is the oracle function that finds a
# def oracle_f_11(qc):
#     # Flips Qubit 2 to |1>, than applies CCZ
#     qc.x(2)
#     qc.ccz(0, 1, 2)


#This is the oracle function that finds a
def oracle_f_11(qc):
    # Applies CCX : this is the Oracle AND function
    qc.ccx(0, 1, 2)

def grover_diffuser(qc):
    qc.h([0, 1])
    qc.x([0, 1])
    qc.cz(0, 1)
    qc.x([0, 1])
    qc.h([0, 1])
    

In [4]:
from qiskit import *

m = 1 # Number of iterations (hardcode for now, but needs to be dynamic)

# Create a quantum circuit with 2 qubits
grover_circuit = QuantumCircuit(3)
grover_circuit.h([0, 1]) # Apply Hadamard gates to the first two qubits

#Apply X, than H to the last qubit
grover_circuit.x(2)
grover_circuit.h(2)

grover_circuit.barrier() # Add a barrier for better visualization

for _ in range(m):
    oracle_f_11(grover_circuit) # Apply the oracle function
    grover_diffuser(grover_circuit) # Apply the diffuser

grover_circuit.barrier() # Add a barrier for better visualization

grover_circuit.measure_all() # Measure all qubits

In [5]:
from qiskit.primitives import StatevectorSampler

# 1. Initialize the modern sampler
sampler = StatevectorSampler()

# 2. Run your circuit (enclosed in a list) to get the job object
job = sampler.run([grover_circuit])

# 3. Grab the results from the finished job
result = job.result()

# 4. Extract the measurement data for your specific circuit
# [0] targets the first (and only) circuit we passed to the sampler
pub_result = result[0]
counts = pub_result.data.meas.get_counts()

# 5. Print the raw data to see the winning combinations!
print("Measurement Results (State: Counts):")
print(counts)

# ==============================================================================
# NOTE ON OUTPUT INTERPRETATION:
# Qiskit uses little-endian ordering (reads right-to-left: |q2 q1 q0>).
# - The leftmost bit is Qubit 2 (the ancilla).
# - The rightmost two bits are Qubits 1 and 0 (our input search space).
#
# Our raw results show {'011': 483, '111': 541}. 
# This means for 1024/1024 shots, the input qubits are always '11'.
# The ancilla splits 50/50 because it was left in the |-> state.
# Conclusion: The search successfully isolated the winner '11' with 100% accuracy.
# ==============================================================================

Measurement Results (State: Counts):
{'011': 485, '111': 539}


In [6]:
#######################################################
###        GENERALIZATION FOR N QBITS - AND         ###

N = 10 # Number of input qubits (excluding ancilla)
n = N

#This is the oracle function
def oracle_f_and_nqbits(qc):
    n = qc.num_qubits - 1
        
    # Applies CCX : this is the Oracle AND function
    controls = list(range(n)) # Control qubits are 0 to n-1
    target = n

    qc.mcx(controls, target) # Multi-controlled X (generalized CCX)

# ==========================================================================
# WHY THE H-GATES around the mcx:
# Qiskit doesn't have a native 'mcz' (Multi-Controlled Z) method on circuits.
# Because H * X * H = Z, wrapping the target qubit of an 'mcx' gate with 
# Hadamard gates converts it into an 'mcz' gate.
# ==========================================================================
def grover_diffuser_nqbits(qc):
    num_inputs = qc.num_qubits - 1
    
    qc.h(list(range(num_inputs))) 
    qc.x(list(range(num_inputs)))

    controls = list(range(num_inputs-1))
    target  = num_inputs-1

    #This implements a mcz
    qc.h(target)
    qc.mcx(controls, target)
    qc.h(target)

    qc.x(list(range(num_inputs)))
    qc.h(list(range(num_inputs)))

In [7]:
import numpy as np

m = int(np.floor((np.pi/4) * np.sqrt(2**n)))# Number of iterations

# Create a quantum circuit with 2 qubits
grover_circuit = QuantumCircuit(n+1) #Add the ancilla qbit
grover_circuit.h(range(n)) # Apply Hadamard gates to the first n qubits

#Apply X, than H to the last qubit (the ancilla)
grover_circuit.x(n)
grover_circuit.h(n)

grover_circuit.barrier() # Add a barrier for better visualization

for _ in range(m):
    oracle_f_and_nqbits(grover_circuit) # Apply the oracle function
    grover_diffuser_nqbits(grover_circuit) # Apply the diffuser

grover_circuit.barrier() # Add a barrier for better visualization

grover_circuit.measure_all() # Measure all qubits     

In [8]:
from qiskit_aer.backends import AerSimulator
from qiskit import transpile

backend = AerSimulator()
qc_t = transpile(grover_circuit, backend)
job = backend.run(qc_t, shots=1024)
counts = job.result().get_counts(qc_t)
print(counts)

{'01111111111': 535, '11111111111': 489}


In [9]:
### Recherce d'un élément dans une liste non ordonnée

#Creation de la liste et encodage en binaire
import numpy as np
import pandas as pd

#Creation
n = 3
N = 2**n
M = 50

np.random.seed(42)
random_numbers = np.random.randint(0, M, size=N)

df = pd.DataFrame({"Value" : random_numbers})

#Encodage
max_val = df["Value"].max()
value_bit_len = int(np.ceil(np.log2(max_val + 1)))
index_bit_len = n

df["Index"] = df.index
df["Index_Binary"] = df["Index"].apply(lambda x: format(x, f"0{index_bit_len}b"))
df["Value_Binary"] = df["Value"].apply(lambda x: format(x, f"0{value_bit_len}b"))

df = df [["Index", "Index_Binary", "Value", "Value_Binary"]]

# Display the result
print(f"Max Value found: {max_val} (Requires {value_bit_len} bits)")
print(f"List Size: {N} (Requires {index_bit_len} bits)")
print("\nPrepared DataFrame for Grover's Search:")
print(df.to_string(index=False))

Max Value found: 42 (Requires 6 bits)
List Size: 8 (Requires 3 bits)

Prepared DataFrame for Grover's Search:
 Index Index_Binary  Value Value_Binary
     0          000     38       100110
     1          001     28       011100
     2          010     14       001110
     3          011     42       101010
     4          100      7       000111
     5          101     20       010100
     6          110     38       100110
     7          111     18       010010


In [10]:
#Implementation of Quantum State Enconding Function
#Cette fonction encode chaque index en bianaire vers la valeur dans la liste, en binaire également

def encode_index_to_value(qc, df, n_index_bits, reverse=0):
    #It assumes that the qc has the correct number of qubits for the indexes and the values
    row_indices = list(df.index)
    if reverse:
        row_indices = row_indices[::-1]
    
    index_qubits = list(range(n_index_bits))

    for idx in row_indices:
        row = df.loc[idx]

        # ctrl_state is parsed as a plain integer (int("110", 2) = 6), so the standard
        # MSB-first binary string maps correctly to qubit indices — no reversal needed.
        # This is because of how qiskit implements : the register is referenced in little-endian, but ctrl_state expect a standard binary string (MSB-first).
        
        idx_bin_qiskit = row["Index_Binary"]
        value_bin_qiskit = row["Value_Binary"][::-1] # Reverse for qiskit little-endian

        #Note : Quantum cx/mcx gates can only have one target
        #To flip more than one qubit, I need to stack more than one cx/mcx gate
        for bit_position, bit in enumerate(value_bin_qiskit):
            if bit == "1":
                target_value_qubit = n_index_bits + bit_position
                qc.mcx(
                    control_qubits = index_qubits,
                    target_qubit = target_value_qubit,
                    ctrl_state = idx_bin_qiskit,
                )
        qc.barrier()
    
def oracle(qc, target_bin_value):
    #This function compares bit per bit a value to a target value
    #If the target is 1, than I must flip the ancilla iff the value is also 1 (so a simple cx is enough)
    #If the target is 0, than I must flip the ancilla iff the value is also 0 (so a stack of x + cx + x is needed)
    qregs_by_name = {reg.name: reg for reg in qc.qregs}
    qr_value = qregs_by_name["val"]
    qr_ancilla = qregs_by_name["anc"]
    qr_target = qregs_by_name["tgt_phase"]

    target_bin_qiskit = target_bin_value[::-1]

    for bit_position, bit in enumerate(target_bin_qiskit):
        if bit == "1":
            qc.cx(qr_value[bit_position], qr_ancilla[bit_position])
        elif bit == "0":
            qc.x(qr_value[bit_position])
            qc.cx(qr_value[bit_position], qr_ancilla[bit_position])
            qc.x(qr_value[bit_position])

    #Phase kickback on the target ancilla si toutes les ancilla intermédiaires sont à 1 (donc valeurs égales bit à bit)
    qc.mcx(list(qr_ancilla), qr_target[0])

    # Et maintenant il faut uncompute en ordre inverse
    # Cette étape est nécessaire pour "isoler" les qubits sur quoi on cherche (l'index) des autres
    # En fait, quand les autres sont tous à zéro, on peut facotriser et donc les ignorer
    # Si on ne fait pas le uncompute, l'application du diffuseur produit des probabilités fausses
    for bit_position, bit in reversed(list(enumerate(target_bin_qiskit))):
        if bit == "1":
            qc.cx(qr_value[bit_position], qr_ancilla[bit_position])
        elif bit == "0":
            qc.x(qr_value[bit_position])
            qc.cx(qr_value[bit_position], qr_ancilla[bit_position])
            qc.x(qr_value[bit_position])

def diffuser_on_a_register(qc, target_register_name):
    qregs_by_name = {reg.name: reg for reg in qc.qregs}
    
    qr_target = qregs_by_name[target_register_name] #Ici ce sera le registre d'index

    target_qubits = list(qr_target)

    qc.h(target_qubits)
    qc.x(target_qubits)

    controls = target_qubits[:-1]
    target = target_qubits[-1]
    
    qc.h(target)
    qc.mcx(controls, target)
    qc.h(target)
    
    qc.x(target_qubits)
    qc.h(target_qubits)
    

    qc.barrier()

In [11]:
from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister
from qiskit_aer.backends import AerSimulator
from qiskit import transpile

def run_grover_circuit(n_idx, qr_index, qr_value, qr_ancilla, qr_target, cr_measure, df, target_value_string, shots=1024, hw_backend=None):

    num_targets = (df["Value_Binary"] == target_value_string).sum()

    qc = QuantumCircuit(qr_index, qr_value, qr_ancilla, qr_target, cr_measure)
    qc.h(qr_index)
    qc.x(qr_target[0])
    qc.h(qr_target[0])
    qc.barrier()

    iterations = max(1, int(np.floor((np.pi / 4) * np.sqrt(N / max(1, num_targets)))))

    for _ in range(iterations):
        encode_index_to_value(qc, df, n_idx, reverse=0)
        oracle(qc, target_value_string)
        encode_index_to_value(qc, df, n_idx, reverse=1)
        diffuser_on_a_register(qc, "idx")

    qc.measure(qr_index, cr_measure)

    if hw_backend is None:
        backend = AerSimulator()
        transpiled_qc = transpile(qc, backend)
        counts = backend.run(transpiled_qc, shots=shots).result().get_counts(transpiled_qc)
    else:
        from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
        from qiskit_ibm_runtime import SamplerV2 as Sampler
        pm          = generate_preset_pass_manager(backend=hw_backend, optimization_level=1)
        isa_circuit = pm.run(qc)
        print(f"Transpiled depth: {isa_circuit.depth()} gates")
        job = Sampler(hw_backend).run([isa_circuit], shots=shots)
        print(f"Job submitted — ID: {job.job_id()}")
        counts = job.result()[0].data.meas.get_counts()

    best_index = max(counts, key=counts.get)
    return best_index, counts


In [13]:
n_idx = index_bit_len
n_val = value_bit_len

qr_index   = QuantumRegister(n_idx, name="idx")
qr_value   = QuantumRegister(n_val, name="val")
qr_ancilla = QuantumRegister(n_val, name="anc")
qr_target  = QuantumRegister(1,     name="tgt_phase")
cr_measure = ClassicalRegister(n_idx, name="meas")

np.random.seed(7)
targets = df["Value_Binary"].sample(5).tolist()

print("=" * 55)
print("  GROVER SEARCH — 5 RANDOM TARGETS")
print("=" * 55)

for i, target_bin in enumerate(targets):
    target_val = int(target_bin, 2)
    matching_indices = df[df["Value_Binary"] == target_bin]["Index"].tolist()

    print(f"\nRun {i+1}/5")
    print(f"  Target value : {target_val:>3}  (binary: {target_bin})")
    print(f"  Expected idx : {matching_indices}")

    best_bin, counts = run_grover_circuit(
        n_idx, qr_index, qr_value, qr_ancilla, qr_target, cr_measure,
        df, target_bin,
    )

    best_idx = int(best_bin, 2)
    correct = best_idx in matching_indices
    print(f"  Found idx    : {best_idx:>3}  (binary: {best_bin})")
    print(f"  Correct      : {correct}")
    print(f"  Counts       : {dict(sorted(counts.items()))}")

print("\n" + "=" * 55)


  GROVER SEARCH — 5 RANDOM TARGETS

Run 1/5
  Target value :  14  (binary: 001110)
  Expected idx : [2]
  Found idx    :   2  (binary: 010)
  Correct      : True
  Counts       : {'000': 9, '001': 4, '010': 965, '011': 7, '100': 11, '101': 5, '110': 9, '111': 14}

Run 2/5
  Target value :  20  (binary: 010100)
  Expected idx : [5]
  Found idx    :   5  (binary: 101)
  Correct      : True
  Counts       : {'000': 7, '001': 2, '010': 6, '011': 11, '100': 8, '101': 971, '110': 5, '111': 14}

Run 3/5
  Target value :  38  (binary: 100110)
  Expected idx : [0, 6]
  Found idx    :   0  (binary: 000)
  Correct      : True
  Counts       : {'000': 516, '110': 508}

Run 4/5
  Target value :  38  (binary: 100110)
  Expected idx : [0, 6]
  Found idx    :   6  (binary: 110)
  Correct      : True
  Counts       : {'000': 506, '110': 518}

Run 5/5
  Target value :  42  (binary: 101010)
  Expected idx : [3]
  Found idx    :   3  (binary: 011)
  Correct      : True
  Counts       : {'000': 6, '001': 1

In [14]:
from qiskit_ibm_runtime import QiskitRuntimeService

service    = QiskitRuntimeService(channel="ibm_quantum_platform")
hw_backend = service.least_busy(operational=True, simulator=False)
print(f"Backend: {hw_backend.name}")

target_bin_hw = df.loc[3, "Value_Binary"]
matching_hw   = df[df["Value_Binary"] == target_bin_hw]["Index"].tolist()
print(f"Target : {int(target_bin_hw, 2)} (binary: {target_bin_hw}), expected at index {matching_hw}")

best_bin_hw, counts_hw = run_grover_circuit(
    n_idx, qr_index, qr_value, qr_ancilla, qr_target, cr_measure,
    df, target_bin_hw, hw_backend=hw_backend,
)

best_idx_hw = int(best_bin_hw, 2)
print(f"Counts : {dict(sorted(counts_hw.items()))}")
print(f"Found  : index {best_idx_hw} (binary: {best_bin_hw})")
print(f"Correct: {best_idx_hw in matching_hw}")


qiskit_runtime_service.__init__:WARNING:2026-06-01 12:50:23,612: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-06-01 12:50:24,161: Loading instance: open-instance, plan: open
qiskit_runtime_service.backends:WARNING:2026-06-01 12:50:25,941: Using instance: open-instance, plan: open


Backend: ibm_marrakesh
Target : 42 (binary: 101010), expected at index [3]
Transpiled depth: 9283 gates
Job submitted — ID: d8em7sro3njc73ev0fog
Counts : {'000': 130, '001': 146, '010': 109, '011': 130, '100': 120, '101': 114, '110': 137, '111': 138}
Found  : index 1 (binary: 001)
Correct: False


In [12]:
import numpy as np
# GAS for Minimum Search
# Representation de g pour dim = n
N_DIM = 3
n = N_DIM

#Random coefficients generation
BOUND_CONST = 10
low_bound = -BOUND_CONST
high_bound = BOUND_CONST + 1 # +1 because of the way np.random.randint works (exclusive of high)

SEED = 42

np.random.seed(SEED)

#Representation matricielle des coefficients de g
rand_matrix = np.random.randint(low=low_bound, high=high_bound, size=(n, n)).astype(float)
g_matrix = np.triu(rand_matrix) # Je génère une matrice triangulaire supérieure pour éviter les redondances (g_ij = g_ji)
                             # Donc, elle n'est pas dans le format standard de représentation d'une fonction quadratique
                             # Mais ce format est plus utile ici

g_matrix_quadratic = (g_matrix + g_matrix.T) / 2.0 #Je garde la forme quadratique pour l'évaluation classique

#Compute K and minimum number of qubits needed to encode g(x)
pos_sum = np.sum(g_matrix[g_matrix > 0])
neg_sum = np.sum(g_matrix[g_matrix < 0])
K = max(abs(pos_sum), abs(neg_sum))

m = 0
while not(-2**m <= K and K <= 2**(m-1)-1):
    m += 1

# ----------------------------------------
# PRINT STATEMENTS
# ----------------------------------------
print("="*50)
print(" QUANTUM MINIMUM SEARCH: INITIALIZATION ")
print("="*50)
print(f"Dimension space (n) : {n} variables")
print(f"Coefficient bounds   : [{low_bound}, {BOUND_CONST}]")
print("-"*50)
print("Function Matrix g(x)")
print(g_matrix)
print("-"*50)
print(f"Sum of Positive Terms (pos_sum) : {pos_sum}")
print(f"Sum of Negative Terms (neg_sum) : {neg_sum}")
print(f"Absolute Boundary (K)            : {K}")
print("-"*50)
print(f"Qubits needed for g(x) value (m) : {m} qubits")
print("="*50)

 QUANTUM MINIMUM SEARCH: INITIALIZATION 
Dimension space (n) : 3 variables
Coefficient bounds   : [-10, 10]
--------------------------------------------------
Function Matrix g(x)
[[-4.  9.  4.]
 [ 0. -3. 10.]
 [ 0.  0.  0.]]
--------------------------------------------------
Sum of Positive Terms (pos_sum) : 23.0
Sum of Negative Terms (neg_sum) : -7.0
Absolute Boundary (K)            : 23.0
--------------------------------------------------
Qubits needed for g(x) value (m) : 6 qubits


In [13]:
num_x_qubits = n
num_y_qubits = n
num_scalar_qubits = m

REG_X_NAME = "x"
REG_Y_NAME = "y"
REG_SCALAR_NAME = "scalar"
REG_ANC_NAME = "anc"
CR_MEAS_NAME = "meas"

In [28]:
def encode_y(qc, y_bitstring, y_register_name=REG_Y_NAME):

    qr_y = next(reg for reg in qc.qregs if reg.name == y_register_name)
    
    for index, bit in enumerate(y_bitstring):
        if int(bit) == 1:
            qc.x(qr_y[index])

#Comparator function for the minimum search
def add_g_as_a_phase(qc, g_matrix, control_reg_name, sign=1):
    # Dynamic register extraction by checking names
    control_reg = next(reg for reg in qc.qregs if reg.name == control_reg_name)
    qr_scalar = next(reg for reg in qc.qregs if reg.name == REG_SCALAR_NAME)
    
    n = len(control_reg) # Dimension of space
    m = len(qr_scalar)  # Qubit needed to compute  

    for i in range(g_matrix.shape[0]):
        for j in range(i, g_matrix.shape[1]):
            coeff = g_matrix[i ,j]
            if coeff == 0 :
                continue
            controls = []
            for ctrl_index in range(1, n) :
                if i == ctrl_index or j == ctrl_index:
                    qubit = (control_reg[ctrl_index-1])
                    if qubit not in controls:
                        controls.append(qubit)   

            # k = 0 : LSB, k = m-1 : MSB
            # Qiskit registers are little-endian: qr_scalar[0] = LSB, qr_scalar[m-1] = MSB
            # (m-k) maps k=0 → largest denominator (smallest rotation, LSB)
            #              k=m-1 → denominator 1 (largest rotation, MSB)
            for k in range(m):
                angle = (2 * np.pi * coeff * sign) / (2 ** (m-k))
                if len(controls) == 0:
                    qc.p(angle, qr_scalar[k])
                else:
                    qc.mcp(angle, controls, qr_scalar[k])
            
            controls = []

def g_comparator(qc, g_matrix, add_g_as_a_phase=add_g_as_a_phase, reverse=False):
    """
    Computes the difference between g(x) and g(y) into the scalar register.
    By default (reverse=False): computes g(x) - g(y)
    If reverse=True:           computes g(y) - g(x)
    """
    if not(reverse):    
        add_g_as_a_phase(qc, g_matrix, REG_X_NAME)          # + g(x)
        add_g_as_a_phase(qc, g_matrix, REG_Y_NAME, sign=-1) # - g(y)
    else:
        add_g_as_a_phase(qc, g_matrix, REG_Y_NAME)          # + g(y)
        add_g_as_a_phase(qc, g_matrix, REG_X_NAME, sign=-1) # - g(x)


def compute_g_de_x(g_matrix_quadratic, x_bitstring):
    # x0 = 1 always; x_bitstring contains only the n-1 free variables
    x = np.insert(x_bitstring, 0, 1)
    x = np.array(x).reshape(-1, 1)
    g_de_x = (x.T @ g_matrix_quadratic @ x)[0, 0]
    return g_de_x


from qiskit.circuit.library import QFTGate

def apply_DH_oracle(qc, y_bitstring, g_matrix, m_qubits):
    
    qr_scalar  = next(reg for reg in qc.qregs if reg.name == REG_SCALAR_NAME)
    qr_ancilla = next(reg for reg in qc.qregs if reg.name == REG_ANC_NAME)

    # Step 0 : Encode y
    encode_y(qc, y_bitstring)
    
    # Step 1 : H wall on the scalar register
    qc.h(qr_scalar)

    # Step 2 : Compute g(x) - g(y) as a phase
    g_comparator(qc, g_matrix, reverse=False)

    # Step 3 : Inverse QFT  (QFTGate takes only num_qubits, swaps are on by default)
    qc.append(QFTGate(m_qubits).inverse(), qr_scalar)

    # Step 4 : Phase kickback — MSB carries the sign in two's complement
    # Qiskit little-endian: qr_scalar[m_qubits-1] is the MSB
    qc.cx(qr_scalar[m_qubits - 1], qr_ancilla[0])

    # Step 5 : Forward QFT
    qc.append(QFTGate(m_qubits), qr_scalar)

    # Step 6 : Uncompute g(x) - g(y)
    g_comparator(qc, g_matrix, reverse=True)

    # Step 7 : H wall (restores scalar register to |0...0⟩)
    qc.h(qr_scalar)

    # Step 8 : Decode y
    encode_y(qc, y_bitstring)


In [29]:
# First trial
# Je vais prendre des fonctions qui n'admettent qu'un seul minimum
# Matrice Définie Positive : Je prend des diagonales dominantes
# Rq : ça ne resoud pas le pb de l'algo puisqu'il est itératif
# Donc, même si la solution est unique, pdt le déroulement de l'algo je vais quand-même
# trouver plusieurs x qui satisfont g(x) < g(y).

# Je vais quand même tester voir
# Donc je choisi g à diagonale strictement dominante
import numpy as np

# Representation de g pour dim = n
N_DIM = 3

# Random coefficients generation
BOUND_CONST = 10

def generate_quadratic_problem(n=N_DIM, seed=SEED, bound_const=BOUND_CONST):
    low_bound = -BOUND_CONST
    high_bound = BOUND_CONST + 1
    
    np.random.seed(seed)  
    
    low_bound = - bound_const
    high_bound = bound_const + 1

    rng = np.random.default_rng(seed)

    rand_matrix = rng.integers(low=low_bound, high=high_bound, size=(n,n)).astype(float)
    g_matrix = np.triu(rand_matrix)

    # Make it diagonale dominante
    for i in range (n):
        row_sum_abs = np.sum(np.abs(g_matrix[i, i+1:]))

        new_diag_val = row_sum_abs + 1

        # Keep original sign
        sign = np.sign(g_matrix[i,i]) if g_matrix[i,i] != 0 else 1.0
        new_diag_val = sign * new_diag_val
        g_matrix[i,i] = new_diag_val

    g_matrix_quadratic = (g_matrix + g_matrix.T) / 2.0

    pos_sum = np.sum(g_matrix[g_matrix > 0])
    neg_sum = np.sum(g_matrix[g_matrix < 0]) 

    K = abs(max(pos_sum, neg_sum))

    m = 0
    while not( - (2 ** m) <= K and K <= (2**(m-1)-1 ) ):
        m += 1
    
    return g_matrix, g_matrix_quadratic, m, pos_sum, neg_sum, K

def print_quadratic_problem(n, g_matrix, pos_sum, neg_sum, K, m, bound_const):
  
    low_bound = -bound_const
    
    print("="*50)
    print(f"Dimension space (n) : {n} variables")
    print(f"Coefficient bounds   : [{low_bound}, {bound_const}]")
    print("-"*50)
    print("Function Matrix g(x) (Upper Triangular & Diagonal Dominant)")
    print(g_matrix)
    print("-"*50)
    print(f"Sum of Positive Terms (pos_sum) : {pos_sum}")
    print(f"Sum of Negative Terms (neg_sum) : {neg_sum}")
    print(f"Absolute Boundary (K)            : {K}")
    print("-"*50)
    print(f"Qubits needed for g(x) value (m) : {m} qubits")
    print("="*50)

SEEDS = [42, 101, 7, 88, 2026]
BOUNDS = [10, 5, 20, 15, 8]
N_DIMS = [3, 4, 5, 6, 4]

for i, (seed, bound, n_dim) in enumerate(zip(SEEDS, BOUNDS, N_DIMS)):
    print(f" TEST CAS N°{i+1} : Seed={seed}, Bound={bound}, n={n_dim} ")
    
    # Génération du problème quadratique
    g_triu, g_quad, m_qubits, pos, neg, K_val = generate_quadratic_problem(
        n=n_dim, 
        seed=seed, 
        bound_const=bound
    )
    
    # Affichage des résultats via ta fonction dédiée
    print_quadratic_problem(n_dim, g_triu, pos, neg, K_val, m_qubits, bound)
    
    # Un petit espace pour séparer proprement la lecture des blocs dans Jupyter
    print("\n" + " " * 50 + "\n")

 TEST CAS N°1 : Seed=42, Bound=10, n=3 
Dimension space (n) : 3 variables
Coefficient bounds   : [-10, 10]
--------------------------------------------------
Function Matrix g(x) (Upper Triangular & Diagonal Dominant)
[[-10.   6.   3.]
 [  0.  -9.   8.]
 [  0.   0.  -1.]]
--------------------------------------------------
Sum of Positive Terms (pos_sum) : 17.0
Sum of Negative Terms (neg_sum) : -20.0
Absolute Boundary (K)            : 17.0
--------------------------------------------------
Qubits needed for g(x) value (m) : 6 qubits

                                                  

 TEST CAS N°2 : Seed=101, Bound=5, n=4 
Dimension space (n) : 4 variables
Coefficient bounds   : [-5, 5]
--------------------------------------------------
Function Matrix g(x) (Upper Triangular & Diagonal Dominant)
[[-10.   5.   2.  -2.]
 [  0.   3.  -1.   1.]
 [  0.   0.  -6.   5.]
 [  0.   0.   0.  -1.]]
--------------------------------------------------
Sum of Positive Terms (pos_sum) : 16.0
Sum of Neg

In [34]:
import itertools
import os
import numpy as np
import matplotlib.cm as cm
import matplotlib.pyplot as plt

OUTPUT_DIR = "plots/quadratic_landscape"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def naive_solve_quadratic(g_matrix_quadratic):
    n = g_matrix_quadratic.shape[0]

    active_n = n - 1  # x0 = 1 always, so only n-1 variables are free

    all_combinations = itertools.product([0, 1], repeat=active_n)

    temp_results = []
    for comb in all_combinations:
        bit_array = np.array(comb)
        x = np.insert(bit_array, 0, 1).reshape(-1, 1)
        g_x = (x.T @ g_matrix_quadratic @ x)[0, 0]
        bitstring_label = "".join(map(str, comb))
        temp_results.append([bitstring_label, g_x, 0])

    np_results = np.array(temp_results, dtype=object)
    min_index = np.argmin(np_results[:, 1].astype(float))
    np_results[min_index, 2] = 1

    return np_results.tolist()


def plot_quadratic_results(results, seed, n_dim, m_qubits, gas_result=None, save=True):
    """
    Plots the energy landscape and overlays the GAS optimization path if provided.
    labels    = [r[0] for r in results] -> e.g., ['00', '01', '10', '11']
    """
    labels    = [r[0] for r in results]
    g_values  = np.array([r[1] for r in results])
    min_flags = np.array([r[2] for r in results])

    min_idx  = np.where(min_flags == 1)[0][0]
    active_n = len(labels[0])
    n_combinations = 2 ** active_n

    subtitle = (
        f"active variables: {active_n}  |  "
        f"combinations to try: {n_combinations}  |  "
        f"qubits for g(x): {m_qubits}"
    )

    # Création d'un dictionnaire pour mapper rapidement une chaîne binaire '010' à son index sur l'axe X
    label_to_idx = {lbl: idx for idx, lbl in enumerate(labels)}

    # -----------------------------------------------------------------
    # CASE 1: 3D PLOT (active_n == 2, meaning n_dim == 3)
    # -----------------------------------------------------------------
    if active_n == 2:
        fig = plt.figure(figsize=(10, 7))
        ax = fig.add_subplot(111, projection="3d")

        x_coords = np.array([int(b[0]) for b in labels])
        y_coords = np.array([int(b[1]) for b in labels])

        norm   = plt.Normalize(g_values.min(), g_values.max())
        colors = cm.get_cmap("RdYlGn_r")(norm(g_values))

        # Paysage naïf
        ax.scatter(x_coords, y_coords, g_values,
                   c=colors, s=200, edgecolors="black", alpha=0.6)

        ax.scatter(x_coords[min_idx], y_coords[min_idx], g_values[min_idx],
                   color="gold", marker="*", s=500, edgecolors="black",
                   zorder=10, label="Global Minimum (Naive)")

        for i, txt in enumerate(labels):
            ax.text(x_coords[i], y_coords[i],
                    g_values[i] + (g_values.max() - g_values.min()) * 0.05,
                    f"|{txt}⟩", ha="center")

        # --- OVERLAY GAS EN 3D ---
        if gas_result is not None:
            # On reconstruit le chemin emprunté par le GAS à partir de son historique
            # Note: history_g_y contient les valeurs successives de g(y)
            # Pour retrouver les coordonnées X et Y associés, on cherche dans nos résultats naïfs
            gas_x = []
            gas_y = []
            gas_z = gas_result["history_g_y"]
            
            for g_val in gas_z:
                # Trouve l'index correspondant à cette valeur d'énergie dans le paysage naïf
                match_idx = np.where(g_values == g_val)[0][0]
                gas_x.append(x_coords[match_idx])
                gas_y.append(y_coords[match_idx])
                
            # On trace la trajectoire
            ax.plot(gas_x, gas_y, gas_z, color="red", linestyle="--", linewidth=2.5, 
                    marker="v", markersize=8, zorder=15, label="GAS Path")
            # Point final trouvé par le GAS
            ax.scatter(gas_x[-1], gas_y[-1], gas_z[-1], color="red", marker="X", 
                       s=250, edgecolors="black", zorder=20, label="GAS Final Found")

        ax.set_title(
            f"3D Energy Space (n-1 = {active_n} active qubits)\n{subtitle}",
            fontsize=11, weight="bold"
        )
        ax.set_xlabel("$x_1$ coordinate")
        ax.set_ylabel("$x_2$ coordinate")
        ax.set_zlabel("$g(x)$ Value")
        ax.set_xticks([0, 1])
        ax.set_yticks([0, 1])
        plt.legend()

    # -----------------------------------------------------------------
    # CASE 2: 2D SEQUENTIAL LANDSCAPE (active_n != 2)
    # -----------------------------------------------------------------
    else:
        fig, ax = plt.subplots(figsize=(max(10, active_n * 1.5), 5))

        # Paysage naïf
        ax.plot(labels, g_values, color="#1f77b4", linestyle="-",
                marker="o", alpha=0.3, label="Naive Energy Path")

        ax.plot(labels[min_idx], g_values[min_idx],
                marker="*", color="gold", markersize=18, zorder=10,
                markeredgecolor="black", label=f"Global Min ({labels[min_idx]})")

        # --- OVERLAY GAS EN 2D ---
        if gas_result is not None:
            gas_x_indices = []
            gas_z = gas_result["history_g_y"]
            
            for g_val in gas_z:
                match_idx = np.where(g_values == g_val)[0][0]
                gas_x_indices.append(match_idx)
            
            # Trace le chemin du GAS par dessus
            ax.plot(gas_x_indices, gas_z, color="red", linestyle="--", linewidth=2,
                    marker="v", markersize=8, zorder=15, label=f"GAS Path ({gas_result['y_found_before_min']} jumps)")
            
            # Marque le point d'arrivée du GAS
            ax.plot(gas_x_indices[-1], gas_z[-1], marker="X", color="red", 
                    markersize=12, markeredgecolor="black", zorder=20, label="GAS Final Found")

        ax.set_title(
            f"2D Binary Search Space Landscape (n-1 = {active_n} active qubits)\n{subtitle}",
            fontsize=11, weight="bold"
        )
        ax.set_xlabel("Ordered Binary Configurations ($x$)")
        ax.set_ylabel("Cost Function Value $g(x)$")
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
        ax.grid(True, linestyle=":", alpha=0.6)
        ax.legend(loc="upper right")

    plt.tight_layout()

    if save:
        filename = f"quadratic_landscape_n{n_dim}_seed{seed}.png"
        filepath = os.path.join(OUTPUT_DIR, filename)
        plt.savefig(filepath, dpi=150, bbox_inches="tight")
        print(f"Saved: {filepath}")

    plt.close()

In [31]:
for i, (seed, bound, n_dim) in enumerate(zip(SEEDS, BOUNDS, N_DIMS)):

    g_triu, g_quad, m_qubits, pos, neg, K_val = generate_quadratic_problem(
        n=n_dim,
        seed=seed,
        bound_const=bound
    )

    results = naive_solve_quadratic(g_quad)
    plot_quadratic_results(results, seed=seed, n_dim=n_dim, m_qubits=m_qubits)

C:\Users\patri\AppData\Local\Temp\ipykernel_31180\797922517.py:63: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colors = cm.get_cmap("RdYlGn_r")(norm(g_values))


Saved: plots/quadratic_landscape\quadratic_landscape_n3_seed42.png
Saved: plots/quadratic_landscape\quadratic_landscape_n4_seed101.png
Saved: plots/quadratic_landscape\quadratic_landscape_n5_seed7.png
Saved: plots/quadratic_landscape\quadratic_landscape_n6_seed88.png
Saved: plots/quadratic_landscape\quadratic_landscape_n4_seed2026.png


In [39]:
################################################################
# Technique Random Exponential Scaling
################################################################

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator

def GAS_solve_quadratic(g_matrix, g_matrix_quadratic, y, n, m, problem_id, lambda_factor=1.2, verbose=False):
    
    g_de_y = compute_g_de_x(g_matrix_quadratic, y)
    
    active_n = n - 1
    search_space_N  = 2 ** (active_n)             # search space size over the n-1 free variables
                                         # cause the first dimension of the vector is always multiplied by 1 (x^0)
    lambda_factor = lambda_factor                  #optimal BBHT scaling factor (= 6/5)
    max_grover_cap = np.sqrt(search_space_N)   # theoretical cap on m

    m_RES         = 1.0  #Upper bound of RES dynamic range from which j is chosen           
    not_converged = True
    backend       = AerSimulator()

    num_x_qubits = active_n
    num_y_qubits = active_n
    num_scalar_qubits = m

    y_found_count = 0 #Pour compter cmb de points intermédiaires ont été trouvé avant le min
    history_g_y = [g_de_y] #Pour garder en mémoire les étapes intermédiaires trouvées par GAS

    while not_converged:
        # --- Draw j uniformly from {0, 1, ..., floor(m)} ---
        # np.random.randint(low, high) draws from {low, ..., high-1}.
        # Setting high = floor(m) + 1 gives exactly {0, 1, ..., floor(m)}.
        j = np.random.randint(0, int(np.floor(m_RES)) + 1)

        # --- Fresh circuit each pass (y and j both change) ---
        # The definition of the registers is needed in the while loop cause
        # otherwhise Qiskit can make errors when reusing registers in a new qc.

        qr_x      = QuantumRegister(num_x_qubits,     name=REG_X_NAME)
        qr_y      = QuantumRegister(num_y_qubits,     name=REG_Y_NAME)
        qr_scalar = QuantumRegister(num_scalar_qubits, name=REG_SCALAR_NAME)
        qr_ancilla = QuantumRegister(1,               name=REG_ANC_NAME)
        cr_meas   = ClassicalRegister(num_x_qubits,   name=CR_MEAS_NAME)  # measure the x register


        qc = QuantumCircuit(qr_x, qr_y, qr_scalar, qr_ancilla, cr_meas)
        
        qc.h(qr_x)  # uniform superposition over all x

        # Préparation du qubit ancilla
        qc.x(qr_ancilla[0])
        qc.h(qr_ancilla[0])

        # Apply j Grover iterations
        for _ in range(j):
            apply_DH_oracle(qc, y, g_matrix, num_scalar_qubits)
            diffuser_on_a_register(qc, REG_X_NAME)

        # Measure the x register into the classical register
        qc.measure(qr_x, cr_meas)

        # --- Run and extract best candidate ---
        transpiled_qc = transpile(qc, backend)
        counts        = backend.run(transpiled_qc, shots=2048).result().get_counts(transpiled_qc)
        x_bin         = max(counts, key=counts.get)

        # Qiskit returns bitstrings in little-endian order (rightmost bit = qubit 0)
        # Reverse to recover [x1, x2, ...] in the same order as y
        x_candidate = np.array([int(b) for b in reversed(x_bin)], dtype=int)

        # --- Classical comparison ---
        g_de_x = compute_g_de_x(g_matrix_quadratic, x_candidate)

        if g_de_x < g_de_y:
            # Success : better point found, reset m and update y
            y_found_count += 1 #On a trouvé un meilleur point intermédiaire
            if verbose:
                print(f"  [SUCCESS] g(x)={g_de_x:.2f} < g(y)={g_de_y:.2f}  |  j={j}  |  x={x_bin}")
            y      = x_candidate
            g_de_y = g_de_x
            history_g_y.append(g_de_y)
            m_RES  = 1.0
        else:
            if m_RES >= max_grover_cap:
                if verbose:
                    print(f"  [STOP] m reached sqrt(N) = {max_grover_cap:.2f}. Best y: g(y) = {g_de_y:.2f}")
                not_converged = False
            else :
                # Failure : scale up m, capped at sqrt(N)
                m_RES = min(lambda_factor * m_RES, max_grover_cap)
            
    if verbose:
        print(f"\nFinal result : y = {y}  |  g(y) = {g_de_y:.4f}")

    instance_dic = {
        "id": problem_id,
        "n_dim": n,
        "best_y": y.tolist(),
        "min_g_y": g_de_y,
        "y_found_before_min": y_found_count,
        "history_g_y": history_g_y # Super utile pour un plot d'optimisation !
    }

    return instance_dic


In [ ]:
# Step 0 : Start with a random y0
# n-1 free variables because x0 = 1 is always fixed in the g(x) evaluation
np.random.seed(42)

for i, (seed, bound, n_dim) in enumerate(zip(SEEDS, BOUNDS, N_DIMS)):

    g_triu, g_quad, m_qubits, pos, neg, K_val = generate_quadratic_problem(
        n=n_dim,
        seed=seed,
        bound_const=bound
    )
    
    results = naive_solve_quadratic(g_quad)
        
    y_init = np.random.randint(low=0, high=2, size=n_dim-1).astype(int)
    
    result_dic = GAS_solve_quadratic(g_triu, g_quad, y_init, n_dim, m_qubits, problem_id=i, verbose=True)
    
    plot_quadratic_results(results, seed=seed, n_dim=n_dim, m_qubits=m_qubits, gas_result = result_dic)


  [SUCCESS] g(x)=-13.00 < g(y)=-8.00  |  j=0  |  x=01
  [STOP] m reached sqrt(N) = 2.00. Best y: g(y) = -13.00

Final result : y = [1 0]  |  g(y) = -13.0000


C:\Users\patri\AppData\Local\Temp\ipykernel_31180\3180631264.py:65: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colors = cm.get_cmap("RdYlGn_r")(norm(g_values))


Saved: plots/quadratic_landscape\quadratic_landscape_n3_seed42.png
  [SUCCESS] g(x)=-10.00 < g(y)=-4.00  |  j=0  |  x=000
  [SUCCESS] g(x)=-12.00 < g(y)=-10.00  |  j=1  |  x=110
  [SUCCESS] g(x)=-13.00 < g(y)=-12.00  |  j=1  |  x=100
  [SUCCESS] g(x)=-14.00 < g(y)=-13.00  |  j=1  |  x=010
  [STOP] m reached sqrt(N) = 2.83. Best y: g(y) = -14.00

Final result : y = [0 1 0]  |  g(y) = -14.0000
Saved: plots/quadratic_landscape\quadratic_landscape_n4_seed101.png
  [SUCCESS] g(x)=67.00 < g(y)=72.00  |  j=1  |  x=1101
  [SUCCESS] g(x)=62.00 < g(y)=67.00  |  j=2  |  x=0010
  [SUCCESS] g(x)=50.00 < g(y)=62.00  |  j=0  |  x=1100
  [SUCCESS] g(x)=33.00 < g(y)=50.00  |  j=1  |  x=0000
  [STOP] m reached sqrt(N) = 4.00. Best y: g(y) = 33.00

Final result : y = [0 0 0 0]  |  g(y) = 33.0000
Saved: plots/quadratic_landscape\quadratic_landscape_n5_seed7.png
  [STOP] m reached sqrt(N) = 5.66. Best y: g(y) = 46.00

Final result : y = [0 0 1 1 0]  |  g(y) = 46.0000
Saved: plots/quadratic_landscape\quadra